<a href="https://colab.research.google.com/github/rhodes-byu/cs-stat-180/blob/main/notebooks/18-ai-nba-stats-gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install the packages

We need:
- `google-genai` for the Gemini API
- `kagglehub` to download the NBA dataset

In [ ]:
# Install the Gemini SDK and the helper package used to download the dataset.
# Remove --quiet if you want to see the full installation log.
!pip install google-genai kagglehub --quiet

print("Packages installed.")

Packages installed.


## 2. Download and clean the Baseball dataset

Each row in this file represents one player-team stint in one season.
Before we ask questions, we clean a few columns so pandas can work with them reliably.

In [ ]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
DATASET_NAME = "nyagami/video-game-ratings-from-imdb"

# --- Download ---
data_dir = Path(kagglehub.dataset_download(DATASET_NAME))

def load_games(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Normalize column names: strip whitespace, lowercase
    df.columns = df.columns.str.strip().str.lower()

    year_col = next(c for c in df.columns if "year" in c)
    df = df.rename(columns={year_col: "year"})

    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df.dropna(subset=["year"]).copy()
    df["year"] = df["year"].astype(int)

    return df


df = load_games(Path("/content/imdb_video_game_rating.csv"))

Using Colab cache for faster access to the 'video-game-ratings-from-imdb' dataset.


## 3. Create your Gemini client

Get an API key from [Google AI Studio](https://aistudio.google.com/app/apikey), then paste it below.

In a real project, you would usually store the key in an environment variable instead of typing it into a notebook.

In [ ]:
import getpass

from google import genai
from google.genai import types

API_KEY = getpass.getpass("Paste your Gemini API key: ")
MODEL = "gemini-2.5-flash"

client = genai.Client(api_key=API_KEY)
print(f"Client ready for {MODEL}.")

Paste your Gemini API key: ··········
Client ready for gemini-2.5-flash.


## 4. Send a very small first prompt

Before we build anything fancy, it helps to see the simplest possible API call.
We send plain text to Gemini and print the plain-text response.

In [ ]:
# This is the smallest useful Gemini request in the notebook.
intro_response = client.models.generate_content(
    model=MODEL,
    contents="In 2 sentences, explain what an API does for a beginner."
)

print(intro_response.text)

An API (Application Programming Interface) lets different computer programs talk to each other, acting like a translator and messenger. When one program needs information or wants to perform an action using another program's services, the API handles the request and delivers the response.


## 5. Give the model a clear description of the dataset

Large language models do not automatically understand your pandas DataFrame.
We need to describe the columns, the date range, and important rules.

This step is part of **prompt engineering**.
A good prompt makes better code and better answers.

In [ ]:
from textwrap import dedent

DATA_DICTIONARY = dedent(
    """
    DataFrame name: video game ratings
    Rows: 12,637
    Meaning of each row: one video game
    Time span: 1948 through 2022

    Important identity columns
    - year:    Year Published

    - title:  video-game name
    - genre:    type of experience provided by the game
    - rating:      rating from 1 to 10 based on how much people liked the game according to online votes
    - votes:   amount of votes provided online
    - directors: People who directed and developed the video game
    - plot: storyline of the video game
    """
).strip()

print(f"Data dictionary ready ({len(DATA_DICTIONARY.split())} words).")

Data dictionary ready (82 words).


## 6. Build the prompt and the local tool

Here is the key idea in this notebook:

1. We ask Gemini a baseball question.
2. Gemini must call a tool named `run_pandas_query`.
3. The tool contains short pandas code.
4. Python runs that code on `batting`.
5. Gemini explains the result in plain English.

This pattern is more reliable than asking the model to answer from memory, because the answer is tied to the actual dataset.

In [ ]:
import traceback

QUESTION_RULES = dedent(
    """

    You are helping a video game enthusiast explore video games with pandas.

    Use the `run_pandas_query` tool exactly once before answering the question.
    The tool code must use only the `df` DataFrame, plus pd and np.
    Assign the final answer object to a variable named result.

    Rules for the tool code:
    - Write 1 to 6 lines of executable Python.

    - If the tool returns an error, explain the error instead of guessing.

    Final answer style: conversational, specific, and short.
    """
).strip()


def build_question_prompt(question: str) -> str:
    """Build the full prompt sent to Gemini for a video game question.

    Args:
        question: The natural-language question the student wants to ask about
            the df dataset.

    Returns:
        A single prompt string containing the tool rules, the data dictionary,
        and the student's question.

    Example:
        prompt = build_question_prompt("What games were published in 1990?")
        print(prompt[:400])
    """
    return dedent(
        f"""
        {QUESTION_RULES}

        Dataset description:
        {DATA_DICTIONARY}

        Student question: {question}
        """
    ).strip()


def show_prompt(question: str) -> None:
    """Print the exact prompt that will be sent to Gemini.

    Args:
        question: The question you plan to pass to `ask(...)`.

    Returns:
        None. This function prints text so you can inspect the prompt.

    Example:
        show_prompt("What game had the lowest rating in the year 2000?")
    """
    print(build_question_prompt(question))


QUERY_FUNCTION = types.FunctionDeclaration(
    name="run_pandas_query",
    description="Run short Python/pandas code against the df DataFrame and return a compact preview of the result.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": (
                    "Executable Python that uses only df, pd, and np. "
                    "Assign the final object to a variable named result."
                ),
            }
        },
        "required": ["code"],
    },
)

QUERY_TOOL = types.Tool(function_declarations=[QUERY_FUNCTION])


def clean_model_code(code: str) -> str:
    """Remove Markdown code fences from model-generated Python.

    Args:
        code: Raw text returned by Gemini. Sometimes the model wraps Python in
            triple backticks before sending it through the tool.

    Returns:
        A cleaned Python string that can be passed to `exec()`.

    Example:
        clean_model_code("```python\nresult = df.head()\n```")
    """
    code = code.strip()
    if code.startswith("```"):
        lines = code.splitlines()
        code = "\n".join(lines[1:])
        if code.endswith("```"):
            code = "\n".join(code.splitlines()[:-1])
    return code.strip()


def format_result_for_model(result) -> str:
    """Convert a pandas result into a short text preview for Gemini.

    Example:
        sample = df[["tittle", "rating"]].head(3)
        print(format_result_for_model(sample))
    """
    if isinstance(result, pd.DataFrame):
        preview = result.head(20).to_string(index=False)
        return f"Result ({len(result)} rows):\n{preview}"

    if isinstance(result, pd.Series):
        preview = result.head(20).to_string()
        return f"Result:\n{preview}"

    return f"Result: {result}"


def run_pandas_query(code: str, debug: bool = False) -> dict:
    """Run model-generated pandas code against the local DataFrame.

    Args:
        code: Short Python code that uses only `df`, `pd`, and `np`, and stores
            the final answer in a variable named `result`.
        debug: If True, print the generated code and any Python traceback.

    Returns:
        A dictionary with `status`, `code`, and `data_context` fields. This is
        the payload sent back to Gemini after the tool executes.

    Example:
        run_pandas_query(
            "result = df[df['tittle'].str.contains('Zelda', case=False, na=False)].head()",
            debug=True,
        )

    Warning:
        This classroom demo uses `exec()`, which can run arbitrary Python code.
        That is convenient for teaching, but unsafe in production.
    """
    code = clean_model_code(code)
    local_vars = {"df": df, "pd": pd, "np": np}

    if debug:
        print("-- Generated pandas code --")
        print(code)
        print()

    try:
        exec(code, {}, local_vars)
        result = local_vars.get("result")

        if result is None:
            return {
                "status": "error",
                "code": code,
                "data_context": "The code ran, but it did not create a variable named `result`.",
            }

        return {
            "status": "ok",
            "code": code,
            "data_context": format_result_for_model(result),
        }

    except Exception as error:
        if debug:
            traceback.print_exc()

        return {
            "status": "error",
            "code": code,
            "data_context": f"Code failed with error: {error}",
        }


FIRST_PASS_CONFIG = types.GenerateContentConfig(
    tools=[QUERY_TOOL],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(
            mode="ANY",
            allowed_function_names=["run_pandas_query"],
        )
    ),
    temperature=0,
)

FINAL_PASS_CONFIG = types.GenerateContentConfig(
    tools=[QUERY_TOOL],
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(mode="NONE")
    ),
    temperature=0.3,
)


def ask(question: str, debug: bool = False) -> None:
    """Ask Gemini a question about the Video Game dataset.

    Args:
        question: A natural-language question such as "What are the names of the best-rated games from the years 2000 to 2026"
        debug: If True, print the pandas code Gemini generated before the final
            explanation is shown.

    Returns:
        None. The function prints Gemini's final explanation to the notebook.

    Example:
        ask("What game was directed by Konrad Tomaszkiewicz", debug=True)

        ask("What games had over 200 votes?")
    """
    prompt = build_question_prompt(question)
    conversation = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt)],
        )
    ]

    first_response = client.models.generate_content(
        model=MODEL,
        contents=conversation,
        config=FIRST_PASS_CONFIG,
    )

    if not first_response.function_calls:
        print(first_response.text)
        return

    tool_call = first_response.function_calls[0]
    tool_code = tool_call.args.get("code", "")
    tool_result = run_pandas_query(tool_code, debug=debug)

    conversation.append(first_response.candidates[0].content)
    conversation.append(
        types.Content(
            role="tool",
            parts=[
                types.Part.from_function_response(
                    name=tool_call.name,
                    response=tool_result,
                )
            ],
        )
    )

    final_response = client.models.generate_content(
        model=MODEL,
        contents=conversation,
        config=FINAL_PASS_CONFIG,
    )

    print(final_response.text)


print("Ready. Try ask('What game had the highest amount of votes in 2020')")

Ready. Try ask('What game had the highest amount of votes in 2020')


## 7. Function reference

The helper functions now include longer docstrings with `Args`, `Returns`, and examples.

Useful ones to inspect:
- `help(load_batting)`
- `help(build_question_prompt)`
- `help(show_prompt)`
- `help(run_pandas_query)`
- `help(ask)`

Quick usage examples:
- `show_prompt("Who led the league in home runs in 1927?")`
- `ask("Compare Babe Ruth and Ted Williams as hitters.", debug=True)`

Reading function documentation is a normal part of learning APIs, so treat these helpers like mini API endpoints inside the notebook.

In [ ]:
show_prompt("Which games are the top 5 best rated?")

You are helping a video game enthusiast explore video games with pandas.

Use the `run_pandas_query` tool exactly once before answering the question.
The tool code must use only the `df` DataFrame, plus pd and np.
Assign the final answer object to a variable named result.

Rules for the tool code:
- Write 1 to 6 lines of executable Python.

- If the tool returns an error, explain the error instead of guessing.

Final answer style: conversational, specific, and short.

        Dataset description:
        DataFrame name: video game ratings
Rows: 12,637
Meaning of each row: one video game
Time span: 1948 through 2022

Important identity columns
- year:    Year Published

- title:  video-game name
- genre:    type of experience provided by the game
- rating:      rating from 1 to 10 based on how much people liked the game according to online votes
- votes:   amount of votes provided online
- directors: People who directed and developed the video game
- plot: storyline of the video game

 

## 8. Ask questions about the NBA data

These are example prompts you can run right away.
Set `debug=True` when you want to see the pandas code Gemini generated for the tool.

In [ ]:
ask("Give me a brief explanation of the 1st top rated video game's plot",debug=True)

-- Generated pandas code --
result = df.sort_values(by='rating', ascending=False).iloc[0]['plot']

The plot of the top-rated video game, The Last of Us, is about experiencing emotional storytelling and unforgettable characters, and it's been rebuilt for the PlayStation 5.


In [ ]:
ask("What video game genre has the most games rated above 9.5?",debug=True)

-- Generated pandas code --
result = df[df['rating'] > 9.5]['genre'].value_counts().idxmax()

The genre with the most games rated above 9.5 is Action, Adventure, Drama.


In [ ]:
ask("Which director has the most games rated above 9? and how are the games called?",debug=True)

-- Generated pandas code --
df_high_rated = df[df['rating'] > 9]
director_game_counts = df_high_rated['directors'].value_counts()
top_director = director_game_counts.index[0]
games_by_top_director = df_high_rated[df_high_rated['directors'] == top_director]['title'].tolist()
result = {"director": top_director, "games": games_by_top_director}

The director with the most games rated above 9 is **Missing**, and their games include *Red Dead Redemption II*, *Grand Theft Auto V*, *Red Dead Redemption*, *Half-Life: Alyx*, *Portal 2*, *Grand Theft Auto: San Andreas*, *Planescape: Torment*, *The Orange Box*, *The Legend of Zelda: Ocarina of Time 3D*, *Disco Elysium - The Final Cut*, *How Zorak Stole X-Mas*, *Mobile Fighter G-Gundam*, *Sonic the Hedgehog Forever*, *Rain World*, *The Walking Dead: The Telltale Definitive Series*, *Grand Theft Auto: Vice City*, *Max Payne*, *Scarface: Money. Power. Respect.*, *GTAV*, *Portal*, *Disco Elysium*, *Gothic II*, *Max Payne 2: The Fall of Max Payne*, and

In [ ]:
ask("Compare Barry Bonds and Hank Aaron",debug=True)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 8.635048691s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}

In [ ]:
ask("Who had the most strikeouts per at-bat in their career?",debug= True)

In [ ]:
ask("Which players had the best OPS in seasons where they had at least 400 at-bats?", debug=True)

## 10. Reflection questions

1. Where in this notebook do we actually make an API request?
2. Which parts of this notebook count as prompt engineering?
3. Why is tool calling better than asking Gemini to answer from memory?
4. What did `debug=True` reveal about the model's pandas code?
5. Why is `exec()` acceptable for a classroom demo but dangerous in production?
6. How could you make this project safer in a real application?